In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 3. Feature Engineering

### 3.1 Feature Classification

The input features are divided into numerical and categorical groups. This classification helps determine which transformations are appropriate for each feature.

The property identifier (`PRT_ID`) is excluded from the predictive features, while `SALES_PRICE` is retained as the target variable.


In [3]:
data_path = Path("../data/processed/chennai_house_price_prediction_cleaned.csv")

df = pd.read_csv(data_path)

In [4]:
df.head()

,PRT_ID,AREA,INT_SQFT,DIST_MAINROAD,N_BEDROOM,N_BATHROOM,N_ROOM,SALE_COND,PARK_FACIL,BUILDTYPE,UTILITY_AVAIL,STREET,MZZONE,QS_ROOMS,QS_BATHROOM,QS_BEDROOM,QS_OVERALL,COMMIS,SALES_PRICE
0,P03210,Karapakkam,1004,131,1.0,1.0,3,AbNormal,Yes,Commercial,AllPub,Paved,A,4.0,3.9,4.9,4.330,144400,7600000
1,P09411,Anna Nagar,1986,26,2.0,1.0,5,AbNormal,No,Commercial,AllPub,Gravel,RH,4.9,4.2,2.5,3.765,304049,21717770
2,P01812,Adyar,909,70,1.0,1.0,3,AbNormal,Yes,Commercial,ELO,Gravel,RL,4.1,3.8,2.2,3.090,92114,13159200
3,P05346,Velachery,1855,14,3.0,2.0,5,Family,No,Others,NoSewr,Paved,I,4.7,3.9,3.6,4.010,77042,9630290
4,P06210,Karapakkam,1226,84,1.0,1.0,3,AbNormal,Yes,Others,AllPub,Gravel,C,3.0,2.5,4.1,3.290,74063,7406250


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7109 entries, 0 to 7108
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PRT_ID         7109 non-null   str    
 1   AREA           7109 non-null   str    
 2   INT_SQFT       7109 non-null   int64  
 3   DIST_MAINROAD  7109 non-null   int64  
 4   N_BEDROOM      7109 non-null   float64
 5   N_BATHROOM     7109 non-null   float64
 6   N_ROOM         7109 non-null   int64  
 7   SALE_COND      7109 non-null   str    
 8   PARK_FACIL     7109 non-null   str    
 9   BUILDTYPE      7109 non-null   str    
 10  UTILITY_AVAIL  7109 non-null   str    
 11  STREET         7109 non-null   str    
 12  MZZONE         7109 non-null   str    
 13  QS_ROOMS       7109 non-null   float64
 14  QS_BATHROOM    7109 non-null   float64
 15  QS_BEDROOM     7109 non-null   float64
 16  QS_OVERALL     7109 non-null   float64
 17  COMMIS         7109 non-null   int64  
 18  SALES_PRICE    7109

In [6]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PRT_ID,7109,7109,P03210,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AREA,7109,7,Chrompet,1702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
INT_SQFT,7109.0,NaN,NaN,NaN,1382.073006,457.410902,500.0,993.0,1373.0,1744.0,2500.0
DIST_MAINROAD,7109.0,NaN,NaN,NaN,99.603179,57.40311,0.0,50.0,99.0,148.0,200.0
N_BEDROOM,7109.0,NaN,NaN,NaN,1.636939,0.802881,1.0,1.0,1.0,2.0,4.0
N_BATHROOM,7109.0,NaN,NaN,NaN,1.21311,0.409534,1.0,1.0,1.0,1.0,2.0
N_ROOM,7109.0,NaN,NaN,NaN,3.688704,1.019099,2.0,3.0,4.0,4.0,6.0
SALE_COND,7109,5,AdjLand,1439,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PARK_FACIL,7109,2,Yes,3587,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BUILDTYPE,7109,4,House,2444,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
X = df.drop(columns=["SALES_PRICE",'PRT_ID'])
y = df["SALES_PRICE"]

In [8]:
# Identify categorical features from X
categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Identify numerical features from X
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nTotal Features:", X.shape[1])

Categorical Features:
['AREA', 'SALE_COND', 'PARK_FACIL', 'BUILDTYPE', 'UTILITY_AVAIL', 'STREET', 'MZZONE']

Numerical Features:
['INT_SQFT', 'DIST_MAINROAD', 'N_BEDROOM', 'N_BATHROOM', 'N_ROOM', 'QS_ROOMS', 'QS_BATHROOM', 'QS_BEDROOM', 'QS_OVERALL', 'COMMIS']

Total Features: 17


### Step 3.2: Decide which features need encoding

Our categorical columns include features such as AREA, SALE_COND, PARK_FACIL, BUILDTYPE, UTILITY_AVAIL, STREET, and MZZONE, based on our earlier discussion.

Before encoding, let's inspect their actual categories.

In [9]:
# Decide which features need encoding

for col in categorical_features:
    print(f"Feature: \n {col}")
    print(f"Unique categories: {X[col].nunique()}")

Feature: 
 AREA
Unique categories: 7
Feature: 
 SALE_COND
Unique categories: 5
Feature: 
 PARK_FACIL
Unique categories: 2
Feature: 
 BUILDTYPE
Unique categories: 4
Feature: 
 UTILITY_AVAIL
Unique categories: 4
Feature: 
 STREET
Unique categories: 3
Feature: 
 MZZONE
Unique categories: 6


### Step 3.4: Split the dataset

3.2 Train-Test Split

The dataset is divided into training and testing subsets before fitting data-dependent preprocessing transformations.

The training dataset is used to learn preprocessing parameters and train the model. The testing dataset is reserved for evaluating the model on unseen data.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Step 3.5: Build the preprocessing pipeline

Now we'll combine the categorical and numerical transformations.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [12]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nOriginal test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (5687, 17)
Processed training shape: (5687, 41)

Original test shape: (1422, 17)
Processed test shape: (1422, 41)
